# `spectra_cc` vs `fastcc`: runtime comparison

`spectra_cc` (`cobra.flux_analysis.spectra_cc`) is a port of the SPECTRA
package's consistency check [1]_, offered as a sibling of `fastcc`. Both
functions solve the same problem — remove the reactions of a model that
cannot carry nonzero flux at steady state — and, as verified in
`tests/test_flux_analysis/test_spectra_cc.py`, they return the *same*
consistent set on every model tested here. This notebook compares how long
each takes to get there.

.. [1] S, P. K., Sridhar, S., Alsmadi, N., Mahadevan, R., & Bhatt, N. P.
       (2026). Generalist method to reconstruct metabolic networks from
       multi-omics data at large-scale. bioRxiv.
       https://doi.org/10.64898/2026.04.02.716249

## Why a difference is expected

Both methods repeatedly solve an LP that tries to push a batch of
not-yet-explained reactions past a flux threshold at once, and remove
whichever ones succeed. The two differ in how they handle a reaction that
resists the bulk LP:

- **`fastcc`** flips the objective sign of the resisting *irreversible*
  reactions and retries the batch LP (`_flip_coefficients`). If that still
  does not explain a reaction, it falls back to solving the LP for
  reactions **one at a time** (the `singletons` phase in `fastcc.py`),
  which is the expensive case: up to one extra LP solve per
  still-unexplained reaction.
- **`spectra_cc`** never needs that fallback. Its forward LP's auxiliary
  variable is unbounded from below, so a reaction that can only carry
  negative flux simply fails to satisfy the forward constraint and is
  picked up by the dedicated reverse LP on the same iteration — no
  flipping, no per-reaction retries.

So the two are expected to be comparable on models where the bulk LP
resolves most reactions right away (little reversibility trouble), and
`spectra_cc` is expected to pull ahead as the reversible, batch-resistant
fraction of the network grows and `fastcc` is forced into its singleton
fallback more often.

In [ ]:
import time
from statistics import mean, stdev

from cobra.io import read_sbml_model
import cobra
import importlib.resources

from cobra.flux_analysis import fastcc, spectra_cc

## Models

The three models bundled with cobrapy itself (`cobra.data`), covering a
small, a medium, and a large network.

In [ ]:
data_dir = importlib.resources.files(cobra.data)

MODELS = {
    "textbook": data_dir / "textbook.xml.gz",
    "salmonella": data_dir / "salmonella.xml.gz",
    "iJO1366": data_dir / "iJO1366.xml.gz",
}

In [ ]:
N_REPEATS = 5


def time_calls(fn, base_model, n, **kwargs):
    """Time n repeats of fn on fresh copies of base_model."""
    times = []
    n_consistent = None
    for _ in range(n):
        m = base_model.copy()
        t0 = time.perf_counter()
        result = fn(m, **kwargs)
        times.append(time.perf_counter() - t0)
        n_consistent = len(result.reactions)
    return times, n_consistent

In [ ]:
print(f'{"model":<12}{"method":<14}{"n_rxns":>8}{"consistent":>12}'
      f'{"mean (s)":>12}{"stdev (s)":>12}{"min (s)":>10}')
print("-" * 80)

for label, path in MODELS.items():
    base = read_sbml_model(str(path))
    base.solver = "glpk"
    n_rxns = len(base.reactions)

    for fn_name, fn, kwargs in (
        ("fastcc", fastcc, {}),
        ("spectra_cc", spectra_cc, {"seed": 0}),
    ):
        times, n_cons = time_calls(fn, base, N_REPEATS, **kwargs)
        print(
            f"{label:<12}{fn_name:<14}{n_rxns:>8}{n_cons:>12}"
            f"{mean(times):>12.4f}{(stdev(times) if len(times) > 1 else 0.0):>12.4f}"
            f"{min(times):>10.4f}"
        )
    print()

# NOTE: n_consistent agreeing between the two methods on every model is the
# correctness check this notebook relies on -- see
# tests/test_flux_analysis/test_spectra_cc.py for the exhaustive version
# (exact agreement with fastcc and find_blocked_reactions, including on
# iJO1366 with zero false positives/negatives).

## Recorded results

Output from the cell above, run once for the record (Python 3.11, glpk solver, 5 repeats per method per model, macOS/arm64, `tol`/`zero_cutoff` left at their defaults):

```
model       method          n_rxns  consistent    mean (s)   stdev (s)   min (s)
--------------------------------------------------------------------------------
textbook    fastcc              95          87      0.0723      0.0109    0.0656
textbook    spectra_cc          95          87      0.0921      0.0022    0.0891

salmonella  fastcc            2891        1686     64.1201      3.9820   60.1992
salmonella  spectra_cc        2891        1686     19.8153      1.0875   18.5464

iJO1366     fastcc            2583        1705     48.6422      3.2148   46.0835
iJO1366     spectra_cc        2583        1705     13.7209      0.4559   13.2427
```

`n_rxns`/`consistent` match between the two methods on every model, as
expected. On the tiny textbook model `fastcc`'s lighter per-iteration
bookkeeping wins by a hair; on both larger, more reversibility-rich
networks `spectra_cc` is **~3.2–3.5x faster**, consistent with it never
needing `fastcc`'s singleton fallback.

**Caveats:** single solver (glpk) and a single machine; no cplex/gurobi
comparison yet. Re-run the cell above to reproduce or extend this.